# 03 · Data augmentation: from about 30 to 350 images per color

**Data augmentation** makes new training images from the ones we have, with small changes: flip, rotate, zoom, brightness and contrast. The model sees more variety, and every color can reach the same number of images, so the class imbalance disappears.

- Each color reaches `TARGET = 350` images: its original photos plus augmented copies, spread evenly among the originals.
- Changes used: horizontal flip, rotation up to ±29°, zoom ±15%, brightness and contrast ±15%.
- **We never change the hue.** A hue change would turn one color into another and change the class.

**Important when testing:** an augmented copy is the same photo with changes, not a new photo. Each copy must stay on the same side (train or test) as its original. That is why the file `manifest.csv` records the original photo and the background of each copy.

Output: `data/helmets_augmented/` (1,750 images). Notebook 04 uses them.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

REPO = Path.cwd().parent                      # the repository folder (notebooks run from notebooks/)
HELMETS = REPO / "data/helmets"
OUTPUTS = REPO / "outputs"; OUTPUTS.mkdir(exist_ok=True)
SIZE = 224                                    # ResNet50 input size in pixels
COLORS = ["green", "orange", "red", "transparent", "yellow"]

meta = pd.read_csv(HELMETS / "helmets_metadata.csv")
# "All image in the Dataset" holds exact copies (same MD5) of "Cartella", so we use only "Cartella"
photos = meta[meta.dataset_folder == "Cartella DATASET Algoritmo convegno"].reset_index(drop=True)
assert photos.md5.is_unique and set(photos.helmet_color) == set(COLORS)

def load(path):
    im = ImageOps.exif_transpose(Image.open(HELMETS / path)).convert("RGB")  # apply camera rotation; drop PNG alpha
    return np.asarray(im.resize((SIZE, SIZE), Image.BILINEAR), dtype=np.uint8)


import keras
from keras import layers

keras.utils.set_random_seed(0)                # same random changes every time we run the notebook
TARGET = 350
DEST = REPO / "data/helmets_augmented"
X = np.stack([load(p) for p in photos.relative_path])

# fill_mode="nearest": the default ("reflect") fills the corners with mirrored pieces of the helmet
augment = keras.Sequential([layers.RandomFlip("horizontal"), layers.RandomRotation(0.08, fill_mode="nearest"),
                            layers.RandomZoom(0.15, fill_mode="nearest"),
                            layers.RandomBrightness(0.15, value_range=(0, 255)), layers.RandomContrast(0.15)])

In [2]:
rows = []
for c in COLORS:
    idx = np.where(photos.helmet_color == c)[0]
    (DEST / c).mkdir(parents=True, exist_ok=True)
    # originals first (copy 0), then augmented copies in turn until TARGET
    sources = np.resize(idx, TARGET)
    copy = np.zeros(len(photos), int)
    for k, i in enumerate(sources):
        img = X[i] if k < len(idx) else augment(X[i][None], training=True)[0].numpy().clip(0, 255).astype("uint8")
        name = f"{Path(photos.filename[i]).stem}_{photos.background[i]}_a{copy[i]:02d}.jpg"
        Image.fromarray(img).save(DEST / c / name, quality=95)
        rows.append({"file": f"{c}/{name}", "color": c, "background": photos.background[i],
                     "original": photos.relative_path[i], "augmented": k >= len(idx)})
        copy[i] += 1

manifest = pd.DataFrame(rows)
manifest.to_csv(DEST / "manifest.csv", index=False)
assert (manifest.color.value_counts() == TARGET).all()
pd.crosstab(manifest.color, manifest.augmented, margins=True).rename(columns={False: "originals", True: "augmented"})

augmented,originals,augmented,All
color,,,
green,28,322,350
orange,31,319,350
red,33,317,350
transparent,30,320,350
yellow,52,298,350
All,174,1576,1750


## One original and 7 of its copies, per color

In [3]:
fig, axes = plt.subplots(len(COLORS), 8, figsize=(14, 9))
for ax in axes.ravel(): ax.set_visible(False)
for i, c in enumerate(COLORS):
    orig = manifest[manifest.color == c].original.iloc[0]
    files = manifest[manifest.original == orig].file.iloc[:8]
    for j, f in enumerate(files):
        ax = axes[i, j]; ax.set_visible(True); ax.imshow(Image.open(DEST / f)); ax.set_xticks([]); ax.set_yticks([])
        if j == 0: ax.set_ylabel(c)
        ax.set_title("original" if j == 0 else f"copy {j}", fontsize=9)
plt.tight_layout(); plt.savefig(OUTPUTS / "03_augmentation_examples.png", dpi=120); plt.show()

Figure not saved in this notebook: it shows the private helmet photos. Run the notebook with the data to see it (also saved in outputs/).
